<a href="https://colab.research.google.com/github/puhspi04/myproject/blob/main/Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🗑️ Plastic Waste Contamination Detection
### Models: YOLOv8 | ResNet-101 (Faster R-CNN) | MobileNetV3 | Hybrid (YOLOv8 + MobileNet)

**Goal:** Detect contaminated plastic → decide ✅ RECYCLABLE or ❌ CONTAMINATED

---
> ⚠️ **First:** Go to `Runtime → Change runtime type → T4 GPU → Save`

## 📦 Section 1 — Install Dependencies

In [6]:
!pip install -q ultralytics albumentations pycocotools seaborn
print('✅ All packages installed')

✅ All packages installed


## ⚙️ Section 2 — Configuration

In [7]:
import os, random, time, json, warnings, shutil
import numpy as np
import torch
import torch.nn as nn
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from pathlib import Path
from collections import defaultdict
warnings.filterwarnings('ignore')

# ── CLASSES (update if your dataset differs) ──────────────────────────────────
CLASSES = ['HDPE Plastic', 'Multi-layer Plastic', 'PET Bottle',
           'Single-Use-Plastic', 'Single-layer Plastic', 'Squeeze-Tube', 'UHT-Box']
NUM_CLASSES = len(CLASSES)

# ── PATHS ─────────────────────────────────────────────────────────────────────
DATA_ROOT   = '/content/drive/MyDrive/Plastic recyclable detection.v3-roboflow-instant-1--eval-.yolov8'
CKPT_DIR    = '/content/checkpoints'
RESULTS_DIR = '/content/results'
YOLO_YAML   = '/content/dataset.yaml'

# ── HYPERPARAMETERS ───────────────────────────────────────────────────────────
YOLO_EPOCHS    = 10
RESNET_EPOCHS  = 10
MOBILE_EPOCHS  = 10
HYBRID_EPOCHS  = 10
BATCH_SIZE     = 16
IMG_SIZE       = 640
CONF_THRESH    = 0.25
RECYCLE_THRESH = 0.5
SEED           = 42
NORM_MEAN      = [0.485, 0.456, 0.406]
NORM_STD       = [0.229, 0.224, 0.225]
CLASS_COLORS   = [(255,0,0),(0,255,0),(0,165,255),(0,0,255),(255,255,0),(255,0,255),(0,255,255)]

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seed()

os.makedirs(CKPT_DIR,    exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'✅ Config ready | Device: {DEVICE.upper()} | Classes: {NUM_CLASSES}')

✅ Config ready | Device: CUDA | Classes: 7


## 📁 Section 3 — Dataset Setup

In [8]:
from google.colab import drive
import yaml

drive.mount('/content/drive', force_remount=True)

# ── Split dataset (test→train since no train labels exist) ────────────────────
os.makedirs(f'{DATA_ROOT}/train/labels', exist_ok=True)

test_imgs = os.listdir(f'{DATA_ROOT}/test/images')
random.shuffle(test_imgs)
train_split = test_imgs[:800]
test_split  = test_imgs[800:]

for fname in train_split:
    stem = os.path.splitext(fname)[0]
    src_img = f'{DATA_ROOT}/test/images/{fname}'
    dst_img = f'{DATA_ROOT}/train/images/{fname}'
    if not os.path.exists(dst_img):
        shutil.copy(src_img, dst_img)
    lbl = f'{stem}.txt'
    src_lbl = f'{DATA_ROOT}/test/labels/{lbl}'
    dst_lbl = f'{DATA_ROOT}/train/labels/{lbl}'
    if os.path.exists(src_lbl) and not os.path.exists(dst_lbl):
        shutil.copy(src_lbl, dst_lbl)

# ── Write fixed YAML ──────────────────────────────────────────────────────────
data = {
    'path':  DATA_ROOT,
    'train': f'{DATA_ROOT}/train/images',
    'val':   f'{DATA_ROOT}/test/images',
    'test':  f'{DATA_ROOT}/test/images',
    'nc':    NUM_CLASSES,
    'names': CLASSES
}
with open(YOLO_YAML, 'w') as f:
    yaml.dump(data, f)

print(f'Train images: {len(os.listdir(f"{DATA_ROOT}/train/images"))}')
print(f'Train labels: {len(os.listdir(f"{DATA_ROOT}/train/labels"))}')
print(f'Test  images: {len(os.listdir(f"{DATA_ROOT}/test/images"))}')
print('✅ Dataset ready!')

Mounted at /content/drive
Train images: 1414
Train labels: 956
Test  images: 990
✅ Dataset ready!


## 🔄 Section 4 — Data Loaders (for ResNet, MobileNet, Hybrid)

In [9]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.backbone_utils import resnet_fpn_backbone
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torch.optim import SGD, Adam
from torch.optim.lr_scheduler import MultiStepLR, CosineAnnealingLR
from torch.cuda.amp import GradScaler, autocast
from tqdm.notebook import tqdm

def get_transforms(train=True):
    if train:
        return A.Compose([
            A.LongestMaxSize(max_size=640),
            A.PadIfNeeded(640, 640, border_mode=cv2.BORDER_CONSTANT, value=114),
            A.HorizontalFlip(p=0.5),
            A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, p=0.5),
            A.Blur(blur_limit=3, p=0.1),
            A.GaussNoise(p=0.1),
            A.Normalize(mean=NORM_MEAN, std=NORM_STD),
            ToTensorV2(),
        ], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels'], min_visibility=0.3))
    else:
        return A.Compose([
            A.LongestMaxSize(max_size=640),
            A.PadIfNeeded(640, 640, border_mode=cv2.BORDER_CONSTANT, value=114),
            A.Normalize(mean=NORM_MEAN, std=NORM_STD),
            ToTensorV2(),
        ], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

class ContaminationDataset(Dataset):
    def __init__(self, img_dir, lbl_dir, transforms=None):
        self.img_dir    = Path(img_dir)
        self.lbl_dir    = Path(lbl_dir)
        self.transforms = transforms
        self.img_paths  = sorted([p for p in self.img_dir.iterdir()
                                   if p.suffix.lower() in {'.jpg','.jpeg','.png'}])
    def __len__(self): return len(self.img_paths)
    def _load_labels(self, p):
        lbl = self.lbl_dir / (p.stem + '.txt')
        boxes, classes = [], []
        if lbl.exists():
            for line in open(lbl):
                parts = line.strip().split()
                if len(parts) == 5:
                    classes.append(int(parts[0]))
                    boxes.append(list(map(float, parts[1:])))
        return np.array(boxes, dtype=np.float32), np.array(classes, dtype=np.int64)
    def __getitem__(self, idx):
        p   = self.img_paths[idx]
        img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
        boxes, cls_ids = self._load_labels(p)
        if self.transforms:
            out     = self.transforms(image=img,
                                      bboxes=boxes.tolist() if len(boxes) else [],
                                      class_labels=cls_ids.tolist() if len(cls_ids) else [])
            img     = out['image']
            boxes   = np.array(out['bboxes'],       dtype=np.float32) if out['bboxes']       else np.zeros((0,4),dtype=np.float32)
            cls_ids = np.array(out['class_labels'], dtype=np.int64)   if out['class_labels'] else np.zeros(0,dtype=np.int64)
        else:
            img = torch.from_numpy(img.transpose(2,0,1)).float()/255.
        return img, {'boxes': torch.tensor(boxes), 'labels': torch.tensor(cls_ids), 'image_id': torch.tensor([idx])}

def collate_fn(batch):
    imgs, tgts = zip(*batch); return list(imgs), list(tgts)

def get_loaders(batch_size=4):
    tr = ContaminationDataset(f'{DATA_ROOT}/train/images', f'{DATA_ROOT}/train/labels', get_transforms(True))
    te = ContaminationDataset(f'{DATA_ROOT}/test/images',  f'{DATA_ROOT}/test/labels',  get_transforms(False))
    trl = DataLoader(tr, batch_size=batch_size, shuffle=True,  collate_fn=collate_fn, num_workers=2, pin_memory=True)
    tel = DataLoader(te, batch_size=1,           shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)
    print(f'✅ Loaders ready | Train: {len(tr)} | Test: {len(te)}')
    return trl, tel

def boxes_yolo_to_xyxy(boxes, w, h):
    if boxes.shape[0] == 0: return boxes
    cx,cy,bw,bh = boxes[:,0],boxes[:,1],boxes[:,2],boxes[:,3]
    return torch.stack([(cx-bw/2)*w,(cy-bh/2)*h,(cx+bw/2)*w,(cy+bh/2)*h],dim=1)

print('✅ Dataset classes defined')

✅ Dataset classes defined


## 🟦 Section 5 — Model 1: YOLOv8
Single-stage detector. Fastest model. Best for real-time use.

In [5]:
from ultralytics import YOLO
os.makedirs(f'{CKPT_DIR}/yolov8', exist_ok=True)

def train_yolov8(epochs=YOLO_EPOCHS):
    print('\n' + '='*55 + '\n  TRAINING: YOLOv8-x\n' + '='*55)
    model = YOLO('yolov8x.pt')   # COCO pretrained → transfer learning
    model.train(
        data=YOLO_YAML, epochs=epochs, batch=16, imgsz=640,
        lr0=0.005, momentum=0.9, weight_decay=0.0005,
        patience=20, optimizer='SGD', mosaic=1.0,
        project=f'{CKPT_DIR}/yolov8', name='run',
        verbose=True, plots=True, exist_ok=True,
    )
    print('✅ YOLOv8 training complete')
    return model

@torch.no_grad()
def yolo_decision(model, image_path, conf=RECYCLE_THRESH):
    results = model.predict(source=image_path, conf=conf, iou=0.5, verbose=False)
    dets = []
    for r in results:
        for box in r.boxes:
            dets.append({'class': CLASSES[int(box.cls[0])], 'class_id': int(box.cls[0]),
                         'confidence': float(box.conf[0]), 'bbox_xyxy': box.xyxy[0].tolist()})
    contaminated = len(dets) > 0
    return {'contaminated': contaminated,
            'decision': '❌ REJECT — Contaminated' if contaminated else '✅ ACCEPT — Recyclable',
            'detections': dets, 'num_contaminants': len(dets)}

yolo_model = train_yolov8()

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

  TRAINING: YOLOv8-x
Ultralytics 8.4.26 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, 

## 🟩 Section 6 — Model 2: ResNet-101 (Faster R-CNN)
Two-stage detector. Most accurate. Uses ResNet-101 backbone — same as your reference paper.

In [10]:
os.makedirs(f'{CKPT_DIR}/resnet', exist_ok=True)

def build_resnet_frcnn(pretrained=True):
    """Faster R-CNN with ResNet-101 FPN backbone."""
    backbone = resnet_fpn_backbone(
        backbone_name='resnet101',
        weights='ResNet101_Weights.IMAGENET1K_V2' if pretrained else None,
        trainable_layers=3)
    anchor_gen = AnchorGenerator(
        sizes=((32,),(64,),(128,),(256,),(512,)),
        aspect_ratios=((0.5,1.0,2.0),)*5)
    model = FasterRCNN(backbone, num_classes=NUM_CLASSES+1,
                       rpn_anchor_generator=anchor_gen,
                       box_score_thresh=0.05, box_nms_thresh=0.5)
    in_feat = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_feat, NUM_CLASSES+1)
    return model

def train_resnet(epochs=RESNET_EPOCHS):
    print('\n' + '='*55 + '\n  TRAINING: ResNet-101 Faster R-CNN\n' + '='*55)
    train_loader, test_loader = get_loaders(batch_size=2)
    model     = build_resnet_frcnn(pretrained=True).to(DEVICE)
    optimizer = SGD([p for p in model.parameters() if p.requires_grad],
                    lr=0.02, momentum=0.9, weight_decay=0.0001)
    scheduler = MultiStepLR(optimizer, milestones=[int(epochs*0.67), int(epochs*0.92)], gamma=0.1)
    scaler    = GradScaler()
    history   = {'train_loss': [], 'val_loss': []}
    best_loss = float('inf')

    for epoch in range(1, epochs+1):
        # ── Train ──
        model.train(); epoch_loss = 0
        pbar = tqdm(train_loader, desc=f'ResNet Epoch {epoch}/{epochs}', leave=False)
        for imgs, tgts in pbar:
            imgs  = [i.to(DEVICE) for i in imgs]
            h, w  = imgs[0].shape[-2:]
            ftgts = [{'boxes': boxes_yolo_to_xyxy(t['boxes'],w,h).to(DEVICE),
                      'labels': (t['labels']+1).to(DEVICE)} for t in tgts]
            optimizer.zero_grad()
            with autocast():
                loss = sum(build_resnet_frcnn if False else model(imgs,ftgts).values())
            scaler.scale(loss).backward()
            nn.utils.clip_grad_norm_(model.parameters(), 10.)
            scaler.step(optimizer); scaler.update()
            epoch_loss += loss.item()
            pbar.set_postfix(loss=f'{loss.item():.3f}')
        scheduler.step()
        avg_tr = epoch_loss / len(train_loader)

        # ── Val ──
        model.train(); val_loss = 0
        with torch.no_grad():
            for imgs, tgts in test_loader:
                imgs  = [i.to(DEVICE) for i in imgs]
                h, w  = imgs[0].shape[-2:]
                ftgts = [{'boxes': boxes_yolo_to_xyxy(t['boxes'],w,h).to(DEVICE),
                          'labels': (t['labels']+1).to(DEVICE)} for t in tgts]
                with autocast():
                    val_loss += sum(model(imgs,ftgts).values()).item()
        avg_vl = val_loss / len(test_loader)
        history['train_loss'].append(avg_tr)
        history['val_loss'].append(avg_vl)
        print(f'  Epoch {epoch:>3} | Train: {avg_tr:.4f} | Val: {avg_vl:.4f} | LR: {scheduler.get_last_lr()[0]:.5f}')
        if avg_vl < best_loss:
            best_loss = avg_vl
            torch.save(model.state_dict(), f'{CKPT_DIR}/resnet/best.pth')
            print(f'    → Best saved ✅')

    print('✅ ResNet-101 training complete')
    return model, history

@torch.no_grad()
def resnet_predict(model, images, conf=CONF_THRESH):
    model.eval()
    preds = model([i.to(DEVICE) for i in images])
    return [{'boxes': p['boxes'][p['scores']>=conf].cpu(),
              'labels': (p['labels'][p['scores']>=conf]-1).cpu(),
              'scores': p['scores'][p['scores']>=conf].cpu()} for p in preds]

def resnet_decision(model, img_tensor, conf=RECYCLE_THRESH):
    preds = resnet_predict(model, [img_tensor], conf)
    dets  = [{'class': CLASSES[int(l)], 'class_id': int(l),
               'confidence': float(s), 'bbox_xyxy': b.tolist()}
             for b,l,s in zip(preds[0]['boxes'],preds[0]['labels'],preds[0]['scores'])
             if 0 <= int(l) < NUM_CLASSES]
    contaminated = len(dets) > 0
    return {'contaminated': contaminated,
            'decision': '❌ REJECT — Contaminated' if contaminated else '✅ ACCEPT — Recyclable',
            'detections': dets, 'num_contaminants': len(dets)}

resnet_model, resnet_history = train_resnet()


  TRAINING: ResNet-101 Faster R-CNN
✅ Loaders ready | Train: 1414 | Test: 990
Downloading: "https://download.pytorch.org/models/resnet101-cd907fc2.pth" to /root/.cache/torch/hub/checkpoints/resnet101-cd907fc2.pth


100%|██████████| 171M/171M [00:01<00:00, 135MB/s]


ResNet Epoch 1/10:   0%|          | 0/707 [00:00<?, ?it/s]

  Epoch   1 | Train: nan | Val: 0.6882 | LR: 0.02000
    → Best saved ✅


ResNet Epoch 2/10:   0%|          | 0/707 [00:00<?, ?it/s]

...

  Epoch   8 | Train: 0.1676 | Val: 0.2445 | LR: 0.00200
    → Best saved ✅


ResNet Epoch 9/10:   0%|          | 0/707 [00:00<?, ?it/s]

  Epoch   9 | Train: 0.1633 | Val: 0.2664 | LR: 0.00020


ResNet Epoch 10/10:   0%|          | 0/707 [00:00<?, ?it/s]

  Epoch  10 | Train: 0.1734 | Val: 0.2636 | LR: 0.00020
✅ ResNet-101 training complete


## 🟧 Section 7 — Model 3: MobileNetV3 (SSD-style detector)
Lightweight model. Fastest inference. Good for edge devices / mobile deployment.

In [15]:
import torchvision
from torchvision.models.detection import ssdlite320_mobilenet_v3_large
from torchvision.models.detection.ssdlite import SSDLiteClassificationHead
from functools import partial

os.makedirs(f'{CKPT_DIR}/mobilenet', exist_ok=True)

def build_mobilenet(pretrained=True):
    """SSDLite with MobileNetV3-Large backbone — lightweight & fast."""
    model = ssdlite320_mobilenet_v3_large(
        weights='SSDLite320_MobileNet_V3_Large_Weights.COCO_V1' if pretrained else None
    )
    in_channels = [
        module.in_channels
        for module in model.head.classification_head.module_list
        if hasattr(module, 'in_channels')
    ]
    num_anchors = model.anchor_generator.num_anchors_per_location()
    model.head.classification_head = SSDLiteClassificationHead(
        in_channels=in_channels,
        num_anchors=num_anchors,
        num_classes=NUM_CLASSES + 1,
        norm_layer=partial(nn.BatchNorm2d, eps=0.001, momentum=0.03)
    )
    return model

def get_loaders(batch_size=4):
    tr = ContaminationDataset(f'{DATA_ROOT}/train/images', f'{DATA_ROOT}/train/labels', get_transforms(True))
    te = ContaminationDataset(f'{DATA_ROOT}/test/images',  f'{DATA_ROOT}/test/labels',  get_transforms(False))
    trl = DataLoader(tr, batch_size=batch_size, shuffle=True,  collate_fn=collate_fn, num_workers=2, pin_memory=True, drop_last=True)
    tel = DataLoader(te, batch_size=batch_size, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True, drop_last=True)
    print(f'✅ Loaders ready | Train: {len(tr)} | Test: {len(te)}')
    return trl, tel

def train_mobilenet(epochs=MOBILE_EPOCHS):
    print('\n' + '='*55 + '\n  TRAINING: MobileNetV3 (SSDLite)\n' + '='*55)
    train_loader, test_loader = get_loaders(batch_size=8)
    model     = build_mobilenet(pretrained=True).to(DEVICE)
    optimizer = Adam([p for p in model.parameters() if p.requires_grad], lr=1e-3, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)
    scaler    = GradScaler()
    history   = {'train_loss': [], 'val_loss': []}
    best_loss = float('inf')

    for epoch in range(1, epochs+1):
        model.train(); epoch_loss = 0
        pbar = tqdm(train_loader, desc=f'MobileNet Epoch {epoch}/{epochs}', leave=False)
        for imgs, tgts in pbar:
            imgs  = [i.to(DEVICE) for i in imgs]
            h, w  = imgs[0].shape[-2:]
            ftgts = [{'boxes': boxes_yolo_to_xyxy(t['boxes'],w,h).to(DEVICE),
                      'labels': (t['labels']+1).to(DEVICE)} for t in tgts]
            optimizer.zero_grad()
            with autocast():
                loss_dict = model(imgs, ftgts)
                loss = sum(loss_dict.values())
            scaler.scale(loss).backward()
            nn.utils.clip_grad_norm_(model.parameters(), 10.)
            scaler.step(optimizer); scaler.update()
            epoch_loss += loss.item()
            pbar.set_postfix(loss=f'{loss.item():.3f}')
        scheduler.step()
        avg_tr = epoch_loss / len(train_loader)

        model.train(); val_loss = 0
        with torch.no_grad():
            for imgs, tgts in test_loader:
                imgs  = [i.to(DEVICE) for i in imgs]
                h, w  = imgs[0].shape[-2:]
                ftgts = [{'boxes': boxes_yolo_to_xyxy(t['boxes'],w,h).to(DEVICE),
                          'labels': (t['labels']+1).to(DEVICE)} for t in tgts]
                with autocast():
                    val_loss += sum(model(imgs,ftgts).values()).item()
        avg_vl = val_loss / len(test_loader)
        history['train_loss'].append(avg_tr)
        history['val_loss'].append(avg_vl)
        print(f'  Epoch {epoch:>3} | Train: {avg_tr:.4f} | Val: {avg_vl:.4f}')
        if avg_vl < best_loss:
            best_loss = avg_vl
            torch.save(model.state_dict(), f'{CKPT_DIR}/mobilenet/best.pth')
            print(f'    → Best saved ✅')

    print('✅ MobileNet training complete')
    return model, history

@torch.no_grad()
def mobilenet_predict(model, images, conf=CONF_THRESH):
    model.eval()
    preds = model([i.to(DEVICE) for i in images])
    return [{'boxes': p['boxes'][p['scores']>=conf].cpu(),
              'labels': (p['labels'][p['scores']>=conf]-1).cpu(),
              'scores': p['scores'][p['scores']>=conf].cpu()} for p in preds]

def mobilenet_decision(model, img_tensor, conf=RECYCLE_THRESH):
    preds = mobilenet_predict(model, [img_tensor], conf)
    dets  = [{'class': CLASSES[int(l)], 'class_id': int(l),
               'confidence': float(s), 'bbox_xyxy': b.tolist()}
             for b,l,s in zip(preds[0]['boxes'],preds[0]['labels'],preds[0]['scores'])
             if 0 <= int(l) < NUM_CLASSES]
    contaminated = len(dets) > 0
    return {'contaminated': contaminated,
            'decision': '❌ REJECT — Contaminated' if contaminated else '✅ ACCEPT — Recyclable',
            'detections': dets, 'num_contaminants': len(dets)}

mobilenet_model, mobilenet_history = train_mobilenet()


  TRAINING: MobileNetV3 (SSDLite)
✅ Loaders ready | Train: 1414 | Test: 990


MobileNet Epoch 1/10:   0%|          | 0/176 [00:00<?, ?it/s]

  Epoch   1 | Train: 6.4276 | Val: 7.1691
    → Best saved ✅


MobileNet Epoch 2/10:   0%|          | 0/176 [00:00<?, ?it/s]

  Epoch   2 | Train: 5.8093 | Val: 6.5641
    → Best saved ✅


...

  Epoch   8 | Train: 5.1962 | Val: 5.4020
    → Best saved ✅


MobileNet Epoch 9/10:   0%|          | 0/176 [00:00<?, ?it/s]

  Epoch   9 | Train: 5.3258 | Val: 5.3983
    → Best saved ✅


MobileNet Epoch 10/10:   0%|          | 0/176 [00:00<?, ?it/s]

  Epoch  10 | Train: 4.7559 | Val: 5.3976
    → Best saved ✅
✅ MobileNet training complete


## 🔴 Section 8 — Model 4: Hybrid Model (YOLOv8 + MobileNetV3)
**Novel contribution** — YOLOv8 detects objects, MobileNetV3 re-classifies each crop.
Best of both: YOLO's speed + MobileNet's classification accuracy.

```
Image → YOLOv8 (detect regions) → crop each region → MobileNetV3 (classify) → final decision
```

In [16]:
import torchvision.models as tvm
import torchvision.transforms as T

os.makedirs(f'{CKPT_DIR}/hybrid', exist_ok=True)

# ── MobileNetV3 Classifier (classifies cropped regions) ──────────────────────
class MobileNetClassifier(nn.Module):
    """MobileNetV3-Small fine-tuned as a crop classifier."""
    def __init__(self, num_classes=NUM_CLASSES, pretrained=True):
        super().__init__()
        weights = tvm.MobileNet_V3_Small_Weights.IMAGENET1K_V1 if pretrained else None
        base    = tvm.mobilenet_v3_small(weights=weights)
        # Replace final classifier
        in_feat = base.classifier[3].in_features
        base.classifier[3] = nn.Linear(in_feat, num_classes)
        self.backbone = base

    def forward(self, x):
        return self.backbone(x)

# ── Classification dataset from YOLO crops ───────────────────────────────────
class CropDataset(Dataset):
    """Crops objects from images using their YOLO labels for classification."""
    def __init__(self, img_dir, lbl_dir, img_size=224):
        self.img_dir  = Path(img_dir)
        self.lbl_dir  = Path(lbl_dir)
        self.img_size = img_size
        self.tf = T.Compose([
            T.ToPILImage(),
            T.Resize((img_size, img_size)),
            T.ToTensor(),
            T.Normalize(mean=NORM_MEAN, std=NORM_STD)
        ])
        # Build crop list
        self.crops = []
        for img_path in sorted(self.img_dir.glob('*.jpg')):
            lbl_path = self.lbl_dir / (img_path.stem + '.txt')
            if not lbl_path.exists(): continue
            img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
            h, w = img.shape[:2]
            for line in open(lbl_path):
                parts = line.strip().split()
                if len(parts) != 5: continue
                cls = int(parts[0])
                cx, cy, bw, bh = map(float, parts[1:])
                x1 = max(0, int((cx-bw/2)*w))
                y1 = max(0, int((cy-bh/2)*h))
                x2 = min(w, int((cx+bw/2)*w))
                y2 = min(h, int((cy+bh/2)*h))
                if x2>x1 and y2>y1:
                    self.crops.append((img[y1:y2, x1:x2].copy(), cls))
        print(f'  CropDataset: {len(self.crops)} crops')

    def __len__(self): return len(self.crops)
    def __getitem__(self, idx):
        crop, cls = self.crops[idx]
        return self.tf(crop), cls

def train_hybrid_classifier(epochs=HYBRID_EPOCHS):
    print('\n' + '='*55 + '\n  TRAINING: Hybrid (YOLOv8 + MobileNetV3 Classifier)\n' + '='*55)

    # Build crop datasets
    train_crops = CropDataset(f'{DATA_ROOT}/train/images', f'{DATA_ROOT}/train/labels')
    test_crops  = CropDataset(f'{DATA_ROOT}/test/images',  f'{DATA_ROOT}/test/labels')

    if len(train_crops) == 0:
        print('⚠️ No crops found — check labels exist'); return None, {}

    train_loader = DataLoader(train_crops, batch_size=32, shuffle=True,  num_workers=2)
    test_loader  = DataLoader(test_crops,  batch_size=32, shuffle=False, num_workers=2)

    model     = MobileNetClassifier(num_classes=NUM_CLASSES).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)
    history   = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_acc  = 0

    for epoch in range(1, epochs+1):
        # ── Train ──
        model.train(); tr_loss=0; tr_correct=0; tr_total=0
        for crops, labels in tqdm(train_loader, desc=f'Hybrid Epoch {epoch}/{epochs}', leave=False):
            crops, labels = crops.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            out  = model(crops)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
            tr_loss    += loss.item()
            tr_correct += (out.argmax(1) == labels).sum().item()
            tr_total   += labels.size(0)
        scheduler.step()
        tr_acc = tr_correct / tr_total

        # ── Val ──
        model.eval(); vl_loss=0; vl_correct=0; vl_total=0
        with torch.no_grad():
            for crops, labels in test_loader:
                crops, labels = crops.to(DEVICE), labels.to(DEVICE)
                out   = model(crops)
                vl_loss    += criterion(out, labels).item()
                vl_correct += (out.argmax(1) == labels).sum().item()
                vl_total   += labels.size(0)
        vl_acc = vl_correct / vl_total

        history['train_loss'].append(tr_loss/len(train_loader))
        history['val_loss'].append(vl_loss/len(test_loader))
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(vl_acc)
        print(f'  Epoch {epoch:>3} | Train Loss: {tr_loss/len(train_loader):.4f} Acc: {tr_acc:.3f} | Val Loss: {vl_loss/len(test_loader):.4f} Acc: {vl_acc:.3f}')
        if vl_acc > best_acc:
            best_acc = vl_acc
            torch.save(model.state_dict(), f'{CKPT_DIR}/hybrid/best_classifier.pth')
            print(f'    → Best classifier saved (acc={best_acc:.3f}) ✅')

    print('✅ Hybrid classifier training complete')
    return model, history

@torch.no_grad()
def hybrid_decision(yolo_model, classifier, image_path, conf=RECYCLE_THRESH):
    """
    Stage 1: YOLOv8 detects all regions
    Stage 2: MobileNetV3 re-classifies each cropped region
    Final:   Combined confidence → ACCEPT or REJECT
    """
    img_bgr = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w    = img_rgb.shape[:2]

    # Stage 1 — YOLO detection
    yolo_results = yolo_model.predict(source=image_path, conf=0.1, iou=0.5, verbose=False)

    tf = T.Compose([
        T.ToPILImage(), T.Resize((224,224)), T.ToTensor(),
        T.Normalize(mean=NORM_MEAN, std=NORM_STD)
    ])
    classifier.eval()
    dets = []

    for r in yolo_results:
        for box in r.boxes:
            x1,y1,x2,y2 = [int(v) for v in box.xyxy[0].tolist()]
            x1,y1 = max(0,x1),max(0,y1); x2,y2 = min(w,x2),min(h,y2)
            if x2<=x1 or y2<=y1: continue

            # Stage 2 — MobileNet re-classification
            crop   = img_rgb[y1:y2, x1:x2]
            tensor = tf(crop).unsqueeze(0).to(DEVICE)
            logits = classifier(tensor)
            probs  = torch.softmax(logits, dim=1)[0]
            cls_id = probs.argmax().item()
            # Combined confidence: YOLO detection × MobileNet classification
            combined_conf = float(box.conf[0]) * float(probs[cls_id])

            if combined_conf >= conf:
                dets.append({'class': CLASSES[cls_id], 'class_id': cls_id,
                             'confidence': combined_conf,
                             'yolo_conf': float(box.conf[0]),
                             'mobile_conf': float(probs[cls_id]),
                             'bbox_xyxy': [x1,y1,x2,y2]})

    contaminated = len(dets) > 0
    return {'contaminated': contaminated,
            'decision': '❌ REJECT — Contaminated' if contaminated else '✅ ACCEPT — Recyclable',
            'detections': dets, 'num_contaminants': len(dets)}

hybrid_classifier, hybrid_history = train_hybrid_classifier()


  TRAINING: Hybrid (YOLOv8 + MobileNetV3 Classifier)
  CropDataset: 974 crops
  CropDataset: 1030 crops
Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 137MB/s]


Hybrid Epoch 1/10:   0%|          | 0/31 [00:00<?, ?it/s]

  Epoch   1 | Train Loss: 0.6923 Acc: 0.772 | Val Loss: 0.4916 Acc: 0.857
    → Best classifier saved (acc=0.857) ✅


Hybrid Epoch 2/10:   0%|          | 0/31 [00:00<?, ?it/s]

...

  Epoch   8 | Train Loss: 0.0044 Acc: 0.999 | Val Loss: 0.0808 Acc: 0.982


Hybrid Epoch 9/10:   0%|          | 0/31 [00:00<?, ?it/s]

  Epoch   9 | Train Loss: 0.0026 Acc: 1.000 | Val Loss: 0.0794 Acc: 0.981


Hybrid Epoch 10/10:   0%|          | 0/31 [00:00<?, ?it/s]

  Epoch  10 | Train Loss: 0.0031 Acc: 1.000 | Val Loss: 0.0778 Acc: 0.982
✅ Hybrid classifier training complete


In [25]:
import shutil
shutil.rmtree(f'{CKPT_DIR}/mobilenet', ignore_errors=True)
import os
os.makedirs(f'{CKPT_DIR}/mobilenet', exist_ok=True)
print("✅ Old checkpoint cleared")

✅ Old checkpoint cleared


In [26]:
from torchvision.models.detection.ssdlite import SSDLiteClassificationHead, SSDLiteRegressionHead

def build_mobilenet(pretrained=True):
    model = ssdlite320_mobilenet_v3_large(
        weights='SSDLite320_MobileNet_V3_Large_Weights.COCO_V1' if pretrained else None
    )
    in_channels = [
        module.in_channels
        for module in model.head.classification_head.module_list
        if hasattr(module, 'in_channels')
    ]
    num_anchors = model.anchor_generator.num_anchors_per_location()
    norm_layer  = partial(nn.BatchNorm2d, eps=0.001, momentum=0.03)
    model.head.classification_head = SSDLiteClassificationHead(
        in_channels=in_channels,
        num_anchors=num_anchors,
        num_classes=NUM_CLASSES + 1,
        norm_layer=norm_layer
    )
    model.head.regression_head = SSDLiteRegressionHead(
        in_channels=in_channels,
        num_anchors=num_anchors,
        norm_layer=norm_layer
    )
    return model

In [27]:
mobilenet_model, mobilenet_history = train_mobilenet(epochs=3)


  TRAINING: MobileNetV3 (SSDLite)
✅ Loaders ready | Train: 1414 | Test: 990


MobileNet Epoch 1/3:   0%|          | 0/176 [00:00<?, ?it/s]

  Epoch   1 | Train: 5.5954 | Val: 7.1005
    → Best saved ✅


MobileNet Epoch 2/3:   0%|          | 0/176 [00:00<?, ?it/s]

  Epoch   2 | Train: 5.5102 | Val: 6.2363
    → Best saved ✅


MobileNet Epoch 3/3:   0%|          | 0/176 [00:00<?, ?it/s]

  Epoch   3 | Train: 5.5561 | Val: 6.0060
    → Best saved ✅
✅ MobileNet training complete


In [29]:
import shutil, os
shutil.rmtree(f'{CKPT_DIR}/mobilenet', ignore_errors=True)
os.makedirs(f'{CKPT_DIR}/mobilenet', exist_ok=True)

# Force rebuild from scratch
del mobilenet_model
torch.cuda.empty_cache()

mobilenet_model, mobilenet_history = train_mobilenet(epochs=3)


  TRAINING: MobileNetV3 (SSDLite)
✅ Loaders ready | Train: 1414 | Test: 990


MobileNet Epoch 1/3:   0%|          | 0/176 [00:00<?, ?it/s]

  Epoch   1 | Train: 6.2285 | Val: 8.0983
    → Best saved ✅


MobileNet Epoch 2/3:   0%|          | 0/176 [00:00<?, ?it/s]

  Epoch   2 | Train: 5.4225 | Val: 6.4805
    → Best saved ✅


MobileNet Epoch 3/3:   0%|          | 0/176 [00:00<?, ?it/s]

  Epoch   3 | Train: 5.1915 | Val: 6.3775
    → Best saved ✅
✅ MobileNet training complete


In [31]:
import shutil, os
shutil.rmtree(f'{CKPT_DIR}/mobilenet', ignore_errors=True)
os.makedirs(f'{CKPT_DIR}/mobilenet', exist_ok=True)
del mobilenet_model
torch.cuda.empty_cache()
print("✅ Cleared")

✅ Cleared


In [2]:
from torchvision.models.detection.ssdlite import SSDLiteClassificationHead, SSDLiteRegressionHead

def build_mobilenet(pretrained=True):
    model = ssdlite320_mobilenet_v3_large(
        weights='SSDLite320_MobileNet_V3_Large_Weights.COCO_V1' if pretrained else None
    )
    in_channels = [
        module.in_channels
        for module in model.head.classification_head.module_list
        if hasattr(module, 'in_channels')
    ]
    num_anchors = model.anchor_generator.num_anchors_per_location()
    norm_layer  = partial(nn.BatchNorm2d, eps=0.001, momentum=0.03)
    model.head.classification_head = SSDLiteClassificationHead(
        in_channels=in_channels,
        num_anchors=num_anchors,
        num_classes=NUM_CLASSES + 1,
        norm_layer=norm_layer
    )
    model.head.regression_head = SSDLiteRegressionHead(
        in_channels=in_channels,
        num_anchors=num_anchors,
        norm_layer=norm_layer
    )
    return model

mobilenet_model, mobilenet_history = train_mobilenet(epochs=3)

NameError: name 'train_mobilenet' is not defined

## 📊 Section 9 — Evaluation & Metrics

In [1]:
# ── mAP computation ───────────────────────────────────────────────────────────
def box_iou(b1, b2):
    a1=(b1[:,2]-b1[:,0])*(b1[:,3]-b1[:,1]); a2=(b2[:,2]-b2[:,0])*(b2[:,3]-b2[:,1])
    ix1=torch.max(b1[:,None,0],b2[None,:,0]); iy1=torch.max(b1[:,None,1],b2[None,:,1])
    ix2=torch.min(b1[:,None,2],b2[None,:,2]); iy2=torch.min(b1[:,None,3],b2[None,:,3])
    inter=(ix2-ix1).clamp(0)*(iy2-iy1).clamp(0)
    return inter/(a1[:,None]+a2[None,:]-inter+1e-6)

def compute_ap(rec,prec):
    r=np.concatenate([[0],rec,[1]]); p=np.concatenate([[0],prec,[0]])
    for i in range(len(p)-2,-1,-1): p[i]=max(p[i],p[i+1])
    idx=np.where(r[1:]!=r[:-1])[0]
    return float(np.sum((r[idx+1]-r[idx])*p[idx+1]))

def compute_map(preds, gts):
    iou_thrs = [0.5+0.05*i for i in range(10)]
    cls_aps  = defaultdict(list)
    for iou_t in iou_thrs:
        all_tp=defaultdict(list); all_sc=defaultdict(list); n_gt=defaultdict(int)
        for pr,gt in zip(preds,gts):
            for c in range(NUM_CLASSES):
                pm=pr['labels']==c; gm=gt['labels']==c
                n_gt[c]+=gm.sum().item()
                pb=pr['boxes'][pm]; sc=pr['scores'][pm]; gb=gt['boxes'][gm]
                if len(pb)==0: continue
                order=sc.argsort(descending=True); pb=pb[order]; sc=sc[order]
                matched=torch.zeros(len(gb),dtype=torch.bool)
                for b in pb:
                    if len(gb)==0: all_tp[c].append(0)
                    else:
                        ious=box_iou(b.unsqueeze(0),gb)[0]
                        bi=ious.max(); bv=ious.argmax()
                        if bi>=iou_t and not matched[bv]: all_tp[c].append(1); matched[bv]=True
                        else: all_tp[c].append(0)
                all_sc[c].extend(sc.tolist())
        for c in range(NUM_CLASSES):
            if n_gt[c]==0 or not all_sc[c]: cls_aps[c].append(0.); continue
            sc=np.array(all_sc[c]); tp=np.array(all_tp[c])
            o=sc.argsort()[::-1]; tp=tp[o]
            tpc=np.cumsum(tp); fpc=np.cumsum(1-tp)
            cls_aps[c].append(compute_ap(tpc/(n_gt[c]+1e-6), tpc/(tpc+fpc+1e-6)))
    res={}; aps=[]
    for c in range(NUM_CLASSES):
        ap=float(np.mean(cls_aps[c])) if cls_aps[c] else 0.
        res[CLASSES[c]]=ap; aps.append(ap)
    ap50=[cls_aps[c][0] if cls_aps[c] else 0. for c in range(NUM_CLASSES)]
    res['mAP@0.5']=float(np.mean(ap50))
    res['mAP@[0.5:0.95]']=float(np.mean(aps))
    return res

def collect_preds(model_fn, loader):
    all_preds, all_gts = [], []
    for imgs, tgts in tqdm(loader, desc='Evaluating', leave=False):
        h, w = imgs[0].shape[-2:]
        for t in tgts:
            bx = t['boxes'].clone()
            if bx.shape[0] > 0:
                x1 = (bx[:,0] - bx[:,2]/2) * w; y1 = (bx[:,1] - bx[:,3]/2) * h
                x2 = (bx[:,0] + bx[:,2]/2) * w; y2 = (bx[:,1] + bx[:,3]/2) * h
                gt_bx = torch.stack([x1, y1, x2, y2], dim=1)
            else:
                gt_bx = torch.zeros((0, 4))
            all_gts.append({'boxes': gt_bx, 'labels': t['labels']})
        with torch.no_grad():
            all_preds.extend(model_fn(imgs))
    return all_preds, all_gts

# ── Run evaluation ────────────────────────────────────────────────────────────
print('\n📊 Evaluating all models...')
_, test_loader = get_loaders(batch_size=1)

# YOLOv8
yv = yolo_model.val(data=YOLO_YAML, conf=CONF_THRESH, iou=0.5, verbose=False)
yolo_metrics = {'mAP@0.5': float(yv.box.map50), 'mAP@[0.5:0.95]': float(yv.box.map),
                'Precision': float(yv.box.mp),   'Recall': float(yv.box.mr)}
for i,c in enumerate(CLASSES):
    if i < len(yv.box.ap50): yolo_metrics[c] = float(yv.box.ap50[i])

# ResNet
resnet_model.eval()
rp, rg = collect_preds(lambda imgs: resnet_predict(resnet_model, imgs), test_loader)
resnet_metrics = compute_map(rp, rg)

# MobileNet
mobilenet_model.eval()
mp, mg = collect_preds(lambda imgs: mobilenet_predict(mobilenet_model, imgs), test_loader)
mobile_metrics = compute_map(mp, mg)

all_metrics = {'YOLOv8-x': yolo_metrics, 'ResNet-101': resnet_metrics, 'MobileNetV3': mobile_metrics}

# ── Hybrid accuracy ───────────────────────────────────────────────────────────
if hybrid_history.get('val_acc'):
    hybrid_acc = max(hybrid_history['val_acc'])
    print(f'\n  Hybrid Model Best Val Accuracy: {hybrid_acc:.4f}')

# ── Print table ───────────────────────────────────────────────────────────────
print('\n' + '='*65)
print(f"  {'Metric':<25} {'YOLOv8-x':>12} {'ResNet-101':>12} {'MobileNetV3':>12}")
print('='*65)
for key in ['mAP@0.5','mAP@[0.5:0.95]'] + CLASSES:
    v1=yolo_metrics.get(key,0); v2=resnet_metrics.get(key,0); v3=mobile_metrics.get(key,0)
    print(f"  {key:<25} {v1:>12.4f} {v2:>12.4f} {v3:>12.4f}")
print('='*65)


📊 Evaluating all models...


NameError: name 'get_loaders' is not defined

## 📈 Section 10 — Comparison Charts

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = ['#2196F3','#4CAF50','#FF5722','#9C27B0']

# ── 1. mAP Bar Chart ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10,5))
models  = list(all_metrics.keys())
map50   = [all_metrics[m].get('mAP@0.5',0)        for m in models]
map595  = [all_metrics[m].get('mAP@[0.5:0.95]',0) for m in models]
x=np.arange(len(models)); w=0.35
b1=ax.bar(x-w/2,map50, w,label='mAP@0.5',        color=PALETTE[0],alpha=0.85)
b2=ax.bar(x+w/2,map595,w,label='mAP@[0.5:0.95]', color=PALETTE[1],alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(models,fontsize=12)
ax.set_ylabel('mAP',fontsize=12)
ax.set_title('Model Comparison — mAP on Contamination Test Set',fontsize=13)
ax.legend(); ax.set_ylim(0,1.0)
for b in [*b1,*b2]: ax.text(b.get_x()+b.get_width()/2,b.get_height()+0.01,f'{b.get_height():.3f}',ha='center',va='bottom',fontsize=9)
plt.tight_layout(); plt.savefig(f'{RESULTS_DIR}/map_comparison.png',dpi=150); plt.show()

# ── 2. Per-Class AP Heatmap ───────────────────────────────────────────────────
data = np.array([[all_metrics[m].get(c,0) for m in models] for c in CLASSES])
fig, ax = plt.subplots(figsize=(10,5))
sns.heatmap(data*100,annot=True,fmt='.1f',cmap='YlOrRd',
            xticklabels=models,yticklabels=CLASSES,vmin=0,vmax=100,ax=ax)
ax.set_title('Per-Class AP (%) — Model Comparison',fontsize=13)
plt.tight_layout(); plt.savefig(f'{RESULTS_DIR}/per_class_heatmap.png',dpi=150); plt.show()

# ── 3. Loss Curves ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1,3,figsize=(16,4))
for ax,(hist,name,col) in zip(axes,[
        (resnet_history,  'ResNet-101',   PALETTE[1]),
        (mobilenet_history,'MobileNetV3', PALETTE[2]),
        (hybrid_history,  'Hybrid',       PALETTE[3])]):
    ax.plot(hist['train_loss'],label='Train Loss',color=col,linewidth=2)
    ax.plot(hist['val_loss'],  label='Val Loss',  color=col,linewidth=2,linestyle='--')
    ax.set_title(f'{name} — Loss Curve',fontsize=11)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.legend()
plt.tight_layout(); plt.savefig(f'{RESULTS_DIR}/loss_curves.png',dpi=150); plt.show()

# ── 4. Hybrid Accuracy Curve ──────────────────────────────────────────────────
if hybrid_history.get('train_acc'):
    fig, ax = plt.subplots(figsize=(7,4))
    ax.plot(hybrid_history['train_acc'],label='Train Acc',color=PALETTE[3],linewidth=2)
    ax.plot(hybrid_history['val_acc'],  label='Val Acc',  color=PALETTE[3],linewidth=2,linestyle='--')
    ax.set_title('Hybrid Model — Classification Accuracy',fontsize=12)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy'); ax.legend(); ax.set_ylim(0,1)
    plt.tight_layout(); plt.savefig(f'{RESULTS_DIR}/hybrid_accuracy.png',dpi=150); plt.show()

print('✅ All charts saved!')

## 🔍 Section 11 — Inference: Show Recycling Decision on Test Images

In [ ]:
def annotate_image(img_bgr, dets, decision, model_name=''):
    img=img_bgr.copy(); h,w=img.shape[:2]
    for det in dets:
        cid=det['class_id']; color=CLASS_COLORS[cid%len(CLASS_COLORS)]
        x1,y1,x2,y2=[int(v) for v in det['bbox_xyxy']]
        x1,y1=max(0,x1),max(0,y1); x2,y2=min(w,x2),min(h,y2)
        cv2.rectangle(img,(x1,y1),(x2,y2),color,2)
        txt=f"{det['class']} {det['confidence']:.2f}"
        (tw,th),_=cv2.getTextSize(txt,cv2.FONT_HERSHEY_SIMPLEX,0.5,1)
        cv2.rectangle(img,(x1,y1-th-6),(x1+tw,y1),color,-1)
        cv2.putText(img,txt,(x1,y1-4),cv2.FONT_HERSHEY_SIMPLEX,0.5,(255,255,255),1)
    bc=(0,0,200) if 'REJECT' in decision else (0,180,0)
    cv2.rectangle(img,(0,0),(w,36),bc,-1)
    cv2.putText(img,f'  {decision}  [{model_name}]',(8,25),cv2.FONT_HERSHEY_SIMPLEX,0.55,(255,255,255),2)
    return img

def run_all_models(image_path):
    img_bgr = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    tf  = A.Compose([A.LongestMaxSize(640),A.PadIfNeeded(640,640,border_mode=cv2.BORDER_CONSTANT,value=114),
                     A.Normalize(mean=NORM_MEAN,std=NORM_STD),ToTensorV2()])
    t   = tf(image=img_rgb); img_tensor = t['image']

    decisions = {
        'YOLOv8-x':   yolo_decision(yolo_model, image_path),
        'ResNet-101':  resnet_decision(resnet_model, img_tensor),
        'MobileNetV3': mobilenet_decision(mobilenet_model, img_tensor),
        'Hybrid':      hybrid_decision(yolo_model, hybrid_classifier, image_path) if hybrid_classifier else {'contaminated':False,'decision':'N/A','detections':[],'num_contaminants':0},
    }

    fig, axes = plt.subplots(1,4,figsize=(22,5))
    for ax,(name,dec) in zip(axes,decisions.items()):
        ann=annotate_image(img_bgr, dec['detections'], dec['decision'], name)
        ax.imshow(cv2.cvtColor(ann,cv2.COLOR_BGR2RGB))
        ax.set_title(f"{name}\n{dec['decision']}\nFound: {dec['num_contaminants']}",fontsize=9)
        ax.axis('off')
    plt.suptitle('Contamination Detection — All Models',fontsize=13,fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/inference_comparison.png',dpi=150)
    plt.show()

    print('\n' + '='*55)
    print('  RECYCLING DECISION')
    print('='*55)
    for name,dec in decisions.items():
        print(f'  {name:<15} → {dec["decision"]}')
    print('='*55)

# Run on a test image
test_imgs = sorted(Path(f'{DATA_ROOT}/test/images').glob('*.jpg'))
if test_imgs:
    run_all_models(str(test_imgs[0]))

## 📤 Section 12 — Upload Your Own Image

In [ ]:
from google.colab import files
uploaded = files.upload()
for fname in uploaded:
    run_all_models(fname)

## 💾 Section 13 — Download Results

In [ ]:
import shutil
with open(f'{RESULTS_DIR}/metrics.json','w') as f:
    json.dump(all_metrics, f, indent=2)
shutil.make_archive('/content/contamination_results','zip', RESULTS_DIR)
print('✅ Zipped!')
from google.colab import files
files.download('/content/contamination_results.zip')